# 03 — Demand Forecasting
**Module C**

Hierarchical time-series forecasting with:
- ETS models at each hierarchy level
- MinT reconciliation
- Bootstrap prediction intervals (80% and 95%)
- Disruption-adjusted forecasting
- MASE and CRPS evaluation metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.forecast import run_module_c, disruption_adjusted_forecast
from src.viz import create_forecast_fan_chart

In [ ]:
results = run_module_c()

## Forecast Fan Charts

In [ ]:
sku_results = results['forecast_results']['sku_results']

# Show first 3 SKUs
for sku_id in list(sku_results.keys())[:3]:
    r = sku_results[sku_id]
    fig = create_forecast_fan_chart(
        r['train_series'], r['test_series'],
        r['forecast'], r['intervals'],
        sku_id=sku_id, category=r['category']
    )
    fig.show()
    print(f'{sku_id}: MASE={r["mase"]:.4f}, CRPS={r["crps"]:.4f}')

## Metrics Summary

In [ ]:
fr = results['forecast_results']
print(f'Average MASE: {fr["avg_mase"]:.4f}')
print(f'Average CRPS: {fr["avg_crps"]:.4f}')
print(f'SKUs forecasted: {len(fr["sku_results"])}')
print(f'MASE < 1.0 (beats naive): {sum(1 for m in fr["mase_scores"] if m < 1.0)} / {len(fr["mase_scores"])}')

## Disruption-Adjusted Forecast Comparison

In [ ]:
demo_sku = list(sku_results.keys())[0]
demo = sku_results[demo_sku]

adjusted = disruption_adjusted_forecast(
    demo['forecast'], demo['category'],
    disruption_type='Red Sea Disruption',
    disruption_start_week=4, disruption_duration=12
)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(y=demo['forecast'], name='Baseline', line=dict(color='#3B82F6')))
fig.add_trace(go.Scatter(y=adjusted, name='Disruption-Adjusted', line=dict(color='#EF4444')))
fig.update_layout(title=f'Baseline vs Disruption-Adjusted — {demo_sku}',
                  xaxis_title='Weeks', yaxis_title='Demand', height=350,
                  plot_bgcolor='#0D0D0D', paper_bgcolor='#0D0D0D',
                  font=dict(color='white'))
fig.show()